# Chapter 35 – Worked examples

The capstone has no worked example of its own: its material is ToOp's output. This notebook reads the files a ToOp
run leaves behind, copied into `baseline/` by `baseline_toop.py` (ToOp `ba1874e`, CPU: the node-breaker example
grid three times with the same seed, and the complex example grid once), with the course modules next to it:
`res_table.py`, `switching_distances.py`, `brief.py` and `hvdc_sensitivity.py`. Nothing here needs ToOp; the
programs that do (`baseline_toop.py`, `ac_check_toop.py`, `sld_toop.py`) run from [lab_toop.ipynb](lab_toop.ipynb).

Run it from this folder (the kernel's working directory must be `src/`).

In [ ]:
import brief
import hvdc_sensitivity
import pandas as pd
import res_table
import switching_distances

pd.set_option('display.width', 200)

## `res.json`: the repertoire's best topologies

`fitness` is minus the weighted sum of the target metrics; `best_topos` holds the best topology of each repertoire
cell (one cell per number of split substations here).

In [ ]:
for path in (res_table.NODE_BREAKER, res_table.COMPLEX):
    res = res_table.load_res(path)
    table = res_table.weighted_sum_check(res)
    print(path.parent.parent.name, 'target metrics', res_table.target_metrics(res))
    print('initial fitness', round(res['initial_fitness'], 3))
    print(table.drop(columns=['worst_k_contingency_cases']).to_string(index=False))
    pick = res_table.for_operator(table, max_splits=1)
    print('at most one split:', pick['actions'], round(pick['fitness'], 3), '\n')

## DC against AC switching distance

The DC stage sums the precomputed `reassignment_distance` of the applied actions; the AC stage counts the
reassignments of the realized topology. `docs/topology_optimizer/metrics.md` calls the difference a bug.

In [ ]:
table = switching_distances.all_pairs()
print(table.to_string(index=False))
switching_distances.plot(table);

## A first operator brief

`brief.brief(run_dir)` turns `unsplit_ac_metrics.json`, the `ac_metrics.json` files and `orao_summary.json` into
the four sections of the chapter's template.

In [ ]:
print(brief.brief(brief.NODE_BREAKER_RUN))

## The HVDC setpoint column (project 7)

A setpoint change of $\Delta P$ on a link from bus $i$ to bus $j$ changes every branch flow by
$\Delta P\,(\mathrm{PTDF}_{\cdot,j} - \mathrm{PTDF}_{\cdot,i})$: a re-solve with the two injections says the same.

In [ ]:
fr, to, b, p = hvdc_sensitivity.six_bus()
worst = max(hvdc_sensitivity.check(fr, to, b, p, i, j) for i in range(6) for j in range(6) if i != j)
print('largest difference over all pairs:', worst)
ptdf = hvdc_sensitivity.ptdf_matrix(fr, to, b, 6)
print('column of a link 2 -> 5:', hvdc_sensitivity.hvdc_column(ptdf, 2, 5).round(4))